# CIDS · 14 — S7: spatial-only twin refiner (control)

Refines the **final memory-jury mask M0** (notebook 12, test Dice 0.7495) **near its boundary only** — no new masks:
`z = z0(M0) + allow · g · delta`, with the correction `delta` zero-initialised (before training the output **is** M0; checked on val at epoch 0) and `allow` = within 8 mm of M0's boundary.

- **This notebook: `MODE = 'spatial'`.** Identical network, but the Haar transform is replaced by a learned stride-2 convolution (same 24 channels, same capacity, no frequency decomposition).
- Notebook 13 runs the frequency refiner; **frequency helps only if 13 beats 14** (on val; test once).
- Training masks = out-of-fold jury masks, randomly grown / shrunk / wobbled (so they look like masks of new patients). Inputs: MRI (3 ch) + M0 + its signed distance — no SegResNet map (in-sample on training patients).
- Best epoch by **val** Dice; **test** once: Dice, HD95, NSD, missed, Wilcoxon + bootstrap CI vs M0.

**Settings:** GPU T4 x2 · Internet ON. **Inputs:** cache · **notebook 12 output** (*Add Input → Your Work → Notebooks → 12_s3f_jury_masks*). **Time:** ~4–6 h (60 epochs, time budget 10 h).

In [ ]:
REPO_URL            = 'https://github.com/Vemuri-BK/cids.git'
GIT_BRANCH          = 'main'
EXPECTED_GIT_COMMIT = ''      # paste the full commit id you pushed

CACHE_ROOT = '/kaggle/input/datasets/kumarvemuri1/mama-mia-preprocessed-cache-2ch'
S3F_DIR    = '/kaggle/input'   # notebook 12 output is found automatically (folder with summary.json + masks/)
CODE_DIR   = '/kaggle/working/cids'
MODE       = 'spatial'
OUT        = f'/kaggle/working/s7_{MODE}'
EPOCHS     = 60
PATHS = [CACHE_ROOT]

In [ ]:
import os, subprocess
if os.path.isdir(f'{CODE_DIR}/.git'):
    subprocess.run(['git', '-C', CODE_DIR, 'fetch', 'origin', GIT_BRANCH], check=True)
    subprocess.run(['git', '-C', CODE_DIR, 'checkout', GIT_BRANCH], check=True)
    subprocess.run(['git', '-C', CODE_DIR, 'pull', 'origin', GIT_BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '-b', GIT_BRANCH, REPO_URL, CODE_DIR], check=True)
if EXPECTED_GIT_COMMIT:
    subprocess.run(['git', '-C', CODE_DIR, 'checkout', EXPECTED_GIT_COMMIT], check=True)
head = subprocess.check_output(['git', '-C', CODE_DIR, 'rev-parse', 'HEAD'], text=True).strip()
assert not EXPECTED_GIT_COMMIT or head == EXPECTED_GIT_COMMIT, (head, EXPECTED_GIT_COMMIT)
print('code at commit', head)
%cd {CODE_DIR}

def sh(cmd):
    """Run a shell step, stream its output, and STOP the notebook if it fails (plain !cmd does not)."""
    print('$', cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='')
    if p.wait():
        raise RuntimeError(f'step failed (exit {p.returncode}): {cmd}')

In [ ]:
sh(f'pip install -q monai medim einops appdirs')
sh(f'pytest -q tests')

In [ ]:
import torch
from pathlib import Path
assert torch.cuda.is_available(), 'No GPU! Settings -> Accelerator -> GPU T4 x2, then restart.'
for q in PATHS:
    print('OK ' if Path(q).exists() else '!! MISSING', q)
assert all(Path(q).exists() for q in PATHS)
hits = [p.parent for p in Path(S3F_DIR).rglob('summary.json') if (p.parent / 'masks').is_dir()]
print('notebook 12 output:', hits)
assert hits, 'attach the notebook 12 output (Add Input -> Your Work -> Notebooks -> 12_s3f_jury_masks)'
S3F_DIR = str(hits[0])

## Smoke test (~5 min): 8 cases per split, 2 epochs
Check: `epoch 0 (= M0): val Dice …` (identity check passed), two epoch lines, a TEST table. Numbers on 8 cases mean nothing.

In [ ]:
sh(f'python scripts/s7_refine.py --mode {MODE} --s3f_dir {S3F_DIR} --cache_root {CACHE_ROOT} --work /kaggle/working/rois_smoke --out /kaggle/working/s7_smoke --limit 8 --epochs 2 --val_every 1')
sh(f'rm -rf /kaggle/working/s7_smoke /kaggle/working/rois_smoke')

## Full run (~4–6 h)
**Read:** `epoch 0 (= M0)` val Dice ≈ 0.774 (= notebook 12). Each validation line shows val Dice vs M0 and how many cases got better / worse. At the end: TEST table (`m0_memory_jury` ≈ 0.7495 vs `refined_…`) and `refined vs M0` (delta, CI, Wilcoxon, NSD change).

In [ ]:
sh(f'python scripts/s7_refine.py --mode {MODE} --s3f_dir {S3F_DIR} --cache_root {CACHE_ROOT} --out {OUT} --epochs {EPOCHS}')

In [ ]:
import json, pandas as pd
print(json.dumps(json.load(open(f'{OUT}/test_summary.json')), indent=2))
print(pd.read_csv(f'{OUT}/train_log.csv').to_string())